# SET-UP

In [1]:
!pip install torch torchvision timm opencv-python matplotlib


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os

In [7]:
base_path = '../Khana_Dataset'
os.listdir(base_path)

['labels.txt', 'dataset.tar', 'taxonomy.csv']

In [9]:
# import tarfile

# with tarfile.open('Khana_Dataset/dataset.tar', 'r') as tar:
#     tar.extractall('./')

In [12]:
# list current directory
print(os.listdir('..'))

# count items inside khana folder
amount = len(os.listdir('../khana'))
print(amount)

# list contents of khana folder
print(os.listdir('../khana'))

['preprocessing.ipynb', 'Khana_Dataset', 'best_model_new.pth', 'checkpoints', '.gitignore', 'scripts', 'khana']
81
['vada pav', 'murukku', 'jalebi', 'biryani', 'misal pav', 'mysore pak', 'masala papad', 'poha', 'grilled sandwich', 'samosa', 'rava dosa', 'balushahi', 'bhindi masala', 'chapati', 'paniyaram', 'banana chips', 'idiyappam', '.DS_Store', 'kheer', 'falooda', 'uttapam', 'chivda', 'aloo gobi', 'aloo mutter', 'papdi chaat', 'solkadhi', 'gujhia', 'chana masala', 'moong dal halwa', 'garlic naan', 'besan laddu', 'kulfi', 'chikki', 'chole bhature', 'masala dosa', 'aloo paratha', 'anda curry', 'rajma chawal', 'phirni', 'hara bhara kabab', 'garlic bread', 'seekh kebab', 'medu vada', 'boondi laddu', 'dal khichdi', 'navratan korma', 'margherita pizza', 'pani puri', 'dhokla', 'fish curry', 'kaju katli', 'idli', 'chicken pizza', 'onion pakoda', 'rasgulla', 'chaas', 'sabudana khichdi', 'steamed momo', 'chicken wings', 'paneer pizza', 'gulab jamun', 'puri bhaji', 'aloo methi', 'ghevar', 'thu

# CLEANING

In [14]:
import os
import shutil

folder = '../khana'

for item in os.listdir(folder):
    if item.startswith('.'):
        path = os.path.join(folder, item)
        
        if os.path.isdir(path):
            shutil.rmtree(path)   # delete folder
        else:
            os.remove(path)       # delete file

In [15]:
fixed_amount = len(os.listdir('../khana'))
print(fixed_amount)

80


In [16]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [ ]:
import os

base = '../khana'
count = 0

for cls in os.listdir(base):
    cls_path = os.path.join(base, cls)

    if not os.path.isdir(cls_path):
        continue

    for file in os.listdir(cls_path):

        # skip hidden files
        if file.startswith('.') or file.startswith('._'):
            continue

        file_path = os.path.join(cls_path, file)

        # 🔥 correct check for extension
        name, ext = os.path.splitext(file)

        if ext == '':
            new_path = file_path + '.jpg'

            try:
                os.rename(file_path, new_path)
                count += 1
            except:
                continue

print(f"Renamed {count} files")

Renamed 75321 files


In [18]:
print(os.listdir('../khana/biryani')[:5])

['jdmrr3nzf8zgmpug4vyu.jpg', 'l7a8whys5lacfcvl97ok.jpg', 'hwhoul3doviipi0xmbkm.jpg', 'bsmkxemojk8xn4hdyf4j.jpg', 'nv4k0fmr1rzmckj4jyxn.jpg']


In [35]:
from PIL import Image
import os

bad = []
for root, _, files in os.walk("../khana"):
    for f in files:
        path = os.path.join(root, f)
        try:
            img = Image.open(path)
            if img.mode != "RGB":
                bad.append((path, img.mode))
        except:
            pass

print(len(bad), "non-RGB images")

265 non-RGB images


# TRAINING PIPELINE

In [19]:
import torch
import random
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split, Subset
import timm
from collections import Counter
import matplotlib.pyplot as plt

/Users/nehapalak/.pyenv/versions/3.9.13/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [21]:
import torch
import random
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split, Subset
import timm
from collections import Counter
import matplotlib.pyplot as plt

In [22]:
# transform = transforms.Compose([
#     transforms.Resize((128,128)),
#     transforms.Lambda(lambda img: img.convert("RGB")),
#     transforms.ToTensor(),
# ])

from torchvision.datasets import ImageFolder
from torchvision import transforms
from PIL import Image

# # Custom loader
# def rgb_loader(path):
#     with open(path, 'rb') as f:
#         img = Image.open(f)
#         return img.convert("RGB")

from PIL import Image

def rgb_loader(path):
    with open(path, 'rb') as f:
        img = Image.open(f)
        img = img.convert("RGBA")  # handle transparency first
        img = img.convert("RGB")   # then remove alpha
        return img

# Clean transform (no need for Lambda now)
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
])

# Use custom loader


In [24]:
dataset = ImageFolder('../khana', transform=transform, loader=rgb_loader)

In [25]:
random.seed(42)

# doing stratified split so that we have balanced classes in both train and val sets

# group indices by class
class_indices = {}

for idx, (_, label) in enumerate(dataset.samples):
    if label not in class_indices:
        class_indices[label] = []
    class_indices[label].append(idx)

train_indices = []
val_indices = []

# split per class
for label, indices in class_indices.items():
    random.shuffle(indices)

    split = int(0.8 * len(indices))

    train_indices.extend(indices[:split])
    val_indices.extend(indices[split:])

In [26]:
train_data = Subset(dataset, train_indices)
val_data = Subset(dataset, val_indices)

In [27]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False, num_workers=2)

In [28]:
train_labels = [dataset.samples[i][1] for i in train_indices]
val_labels = [dataset.samples[i][1] for i in val_indices]

print("Train:", Counter(train_labels))
print("Val:", Counter(val_labels))

Train: Counter({53: 6307, 10: 5311, 41: 4715, 34: 4668, 27: 4584, 29: 4564, 26: 4534, 59: 3539, 71: 3460, 70: 3243, 78: 3205, 52: 2708, 5: 2598, 3: 2524, 39: 2426, 15: 2411, 31: 2369, 13: 2298, 79: 2172, 16: 2036, 54: 1992, 23: 1986, 40: 1968, 14: 1790, 75: 1432, 76: 1305, 32: 1296, 24: 1242, 62: 1186, 21: 1001, 22: 968, 19: 944, 66: 920, 42: 855, 0: 805, 12: 799, 57: 799, 35: 769, 20: 742, 67: 694, 55: 627, 65: 572, 36: 564, 38: 536, 77: 485, 9: 478, 73: 475, 72: 470, 64: 449, 44: 434, 43: 432, 63: 417, 8: 386, 69: 365, 68: 312, 33: 296, 49: 295, 25: 285, 61: 282, 2: 280, 60: 256, 48: 244, 28: 235, 51: 233, 18: 221, 30: 215, 4: 213, 56: 213, 37: 204, 74: 194, 47: 182, 11: 180, 45: 180, 58: 174, 50: 172, 46: 163, 17: 160, 6: 143, 7: 136, 1: 100})
Val: Counter({53: 1577, 10: 1328, 41: 1179, 34: 1167, 27: 1146, 29: 1141, 26: 1134, 59: 885, 71: 866, 70: 811, 78: 802, 52: 677, 5: 650, 3: 631, 39: 607, 15: 603, 31: 593, 13: 575, 79: 544, 16: 509, 54: 499, 23: 497, 40: 492, 14: 448, 75: 358,

In [29]:
# adding weighted loss to handle class imbalance

labels = [label for _, label in dataset.samples]
class_counts = Counter(labels)

weights = torch.zeros(len(dataset.classes))

for i in range(len(dataset.classes)):
    weights[i] = 1.0 / class_counts[i]

weights = weights.to(device)

criterion = torch.nn.CrossEntropyLoss(
    weight=weights,
    label_smoothing=0.1
)

#Loss penalizes rare class mistakes more
#model pays attention to them

# MODEL from huggingface

pretrained on imagenet, 80 output classes for our dataset

https://docs.pytorch.org/vision/main/models/generated/torchvision.models.efficientnet_b0.html

In [30]:
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=80)
model = model.to(device)

In [31]:
# loss + optimizer

criterion = torch.nn.CrossEntropyLoss(
    weight=weights,
    label_smoothing=0.1
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=5e-5,
    weight_decay=1e-4
)

In [32]:
save_dir = 'checkpoints/khana_model'
os.makedirs(save_dir, exist_ok=True)

In [33]:
# adding 'resume from checkpoint' functionality so that if training is interrupted, we can continue from where we left off instead of starting over, which i had to do multiple times, in case i wanted to make any changes

start_epoch = 0
best_acc = 0

checkpoint_path = f"{save_dir}/checkpoint.pth"

if os.path.exists("checkpoint.pth"):
    print("Resuming from checkpoint...")
    checkpoint = torch.load("checkpoint.pth")

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    start_epoch = checkpoint['epoch'] + 1
    best_acc = checkpoint['best_acc']

In [34]:
print("training started !!! \n")
# best_acc = 0
patience = 2
no_improve = 0

for epoch in range(20):

    model.train()
    train_correct, train_total = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_acc = train_correct / train_total

    # validation
    model.eval()
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total

    # save best model
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        no_improve = 0
    else:
        no_improve += 1

    # save checkpoint
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_acc': best_acc
    }, "checkpoint.pth")

    print(f"Epoch {epoch+1} | Train: {train_acc:.4f} | Val: {val_acc:.4f}")

    # adding early stopping functionality so that if the model stops improving for 2 consecutive epochs, we stop training to save time and resources

    if no_improve >= patience:
        print("No improvememt for 2 epochs, therefore stopping early")
        break

print("\nTraining completed!")
print(f"Best Val Accuracy: {best_acc:.4f}")

training started !!! 



Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Users/nehapalak/.pyenv/versions/3.9.13/lib/python3.9/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/Users/nehapalak/.pyenv/versions/3.9.13/lib/python3.9/multiprocessing/spawn.py", line 126, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: Can't get attribute 'rgb_loader' on <module '__main__' (built-in)>


KeyboardInterrupt: 

didnt continue in notebook because of multiprocessing issue (num_worker)

In [ ]:
import torch

# Load the contents (usually a dictionary of weights)
data = torch.load('model.pth', map_location=torch.device('cpu'))

# View the keys (layer names)
print(data.keys())

# To see specific weights (e.g., for the first layer)
first_key = list(data.keys())[0]
print(data[first_key])
